# License

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at:

https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Congestion Dispersion Experiment Evaluation

This Notebook provides analysis tools used as part of the publication titled:

Urban Congestion Relief Experiments Through Routing-App Interventions (*Nature Cities, 2026*).

<br/>

In [Section 2](#scrollTo=Synthetic_data_generation), we generate synthetic data structurally similar to that required by the analysis tools. Then, in [Sections 3](#scrollTo=Analysis_using_Mixed_Linear_Models) and [4](#scrollTo=Bayesian_hierarchical_model_estimates), we employ statistical analysis tools used to infer whether interventions positively affected metrics such as speed and fuel emissions with a high degree of confidence.

<br/>

**Content**

-------------------------

1. [Imports](#scrollTo=Imports)  
2. [Synthetic data generation](#scrollTo=Synthetic_data_generation)  
3. [Analysis using Mixed Linear Models](#scrollTo=Analysis_using_Mixed_Linear_Models)  
  3.1 [Utility functions](#scrollTo=3_1_Utility_functions)  
  3.2 [Statistical inference](#scrollTo=3_2_Statistical_inference)
4. [Bayesian Hierarchical Model Estimates](#scrollTo=Bayesian_hierarchical_model_estimates)  
  4.1 [Utility functions](#scrollTo=4_1_Utility_functions)  
  4.2 [Statistical inference](#scrollTo=4_2_Statistical_inference)

## 1. Imports

Run the below cells to download necessary packages.

In [ ]:
!pip install matplotlib

In [ ]:
!pip install numpy

In [ ]:
!pip install pandas

In [ ]:
!pip install scipy

In [ ]:
!pip install seaborn

In [ ]:
!pip install statsmodels

In [ ]:
!pip install tensorflow

In [ ]:
!pip install tensorflow_probability

In [ ]:
!pip install tqdm

Run the below cell to import all packages used below.

In [ ]:
import collections
from copy import deepcopy
from IPython.display import display
import math
import random
from typing import Any, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
import seaborn as sns
import statsmodels.formula.api as smf
import tensorflow as tf
import tensorflow_probability as tfp
from tensorflow_probability import distributions as tfd
from tensorflow_probability import bijectors as tfb
import tqdm

## 2. Synthetic data generation

In [ ]:
# We set peak morning and afternoon hours to 8-10 am and 3-7pm, respectively.
PEAKTIME_HOURS = [8, 9, 15, 16, 17, 18]


def switchback_status(hours_since_start: int, swtichback_hour) -> bool:
  """Returns the switchback status of the experiment.

  Returns:
    True if treatment, False if control.
  """
  days_since_start = hours_since_start // 24

  if days_since_start % 2 == 0:
    return hours_since_start < swtichback_hour
  else:
    return hours_since_start >= swtichback_hour


def generate_synthetic_data(
    cities_list: list,
    num_days: int,
    switchback_hour: int,
    outcome_baseline: dict[str, float],
    outcome_treatment: dict[str, float],
    outcome_peaktime: dict[str, float],
    outcome_std: dict[str, float],
) -> pd.DataFrame:
  """Generate synthetic data for the experiment analysis tools.

  Args:
    cities_list: A list of cities for which synthetic data is to be generated.
    num_days: The number of days the experiment is run for.
    switchback_hour: The hour of the day (0-23) in which the switchback status
      of the experiment is toggled between control and treatment.
    outcome_baseline: The average for some outcome variable assigned to each
      city during control intervals.
    outcome_treatment: The average for some outcome variable assigned to each
      city during tretment intervals.
    outcome_peaktime: An additional bias assigned to speeds during peaktime
      intervals.
    outcome_std: A standard deviation for some outcome assigned to each city.

  Returns:
    A dataframe containing synthetic data, containing the following columns:

    * city: The name of the city data is sampled from.
    * local_hour: The local hour of day (0-23).
    * str_weekday: The day of the week (0=Monday, 6=Sunday), in string format.
    * is_peak: Whether the local hour is a peaktime hour.
    * switchback_on: Whether the switchback status is on (treatment periods) or
      off (control periods).
    * outcome: The outcome variable, whose effect estimate is being measured
      through the below statistical analysis tools.
  """
  local_hours = [i % 24 for i in range(num_days * 24)]
  local_weekday = [(i // 24) % 7 for i in range(num_days * 24)]
  is_peak = [(i % 24) in PEAKTIME_HOURS for i in range(num_days * 24)]
  switchback_on = [
      switchback_status(i, switchback_hour) for i in range(num_days * 24)]

  num_switchback = [0]
  num_switchback_i = 0
  for i in range(1, len(switchback_on)):
    if switchback_on[i] != switchback_on[i - 1]:
      num_switchback_i += 1
    num_switchback.append(num_switchback_i)

  data = []
  for indx, city in enumerate(cities_list):
    outcomes = [
        # baseline
        + (outcome_treatment[city] if switchback_on[i]
           else outcome_baseline[city])
        # peaktime
        + (outcome_peaktime[city] if is_peak[i] else 0)
        # noise
        + random.gauss(0, outcome_std[city])
        for i in range(len(local_hours))
    ]

    data.append(pd.DataFrame({
        "city": [city] * len(local_hours),
        "local_hour": local_hours,
        "local_weekday": local_weekday,
        "str_weekday": [str(i) for i in local_weekday],
        "is_peak": is_peak,
        "switchback_on": switchback_on,
        "outcome": outcomes,
        "num_switchback": num_switchback,
    }))

  return pd.concat(data).reset_index(drop=True)


def visualize_synthetic_data(data: pd.DataFrame) -> None:
  """Plots the distribution of outcomes as a function of time and city."""
  num_cities = len(data["city"].unique())

  fig, ax = plt.subplots(num_cities // 2, 2, figsize=(18, 6))

  for i, city in enumerate(data["city"].unique()):
    df = data[data["city"] == city]
    ax[i // 2, i % 2].plot(df.outcome)

  fig.supxlabel("Time", fontsize=15)
  fig.supylabel("Outcome", fontsize=15)
  fig.tight_layout()

  plt.show()


# Experiment cities.
cities_list = [
    "atlanta",
    "boston",
    "chicago",
    "los_angeles",
    "miami",
    "new_york",
    "philadelphia",
    "salt_lake_city",
    "san_francisco",
    "seattle",
]

# Randomly select baseline and treatment outcome distributions across cities.
outcome_baseline = {city: random.uniform(20, 30) for city in cities_list}
outcome_treatment = {
    city: outcome_baseline[city] + random.uniform(2, 5) for city in cities_list}
outcome_peaktime = {city: random.uniform(0, 2) for city in cities_list}
outcome_std = {city: random.uniform(0, 1) for city in cities_list}

print("Data generation parameters")
print("--------------------------")
print()
display(pd.DataFrame({
    "city": cities_list,
    "outcome_std": [outcome_std[city] for city in cities_list],
    "outcome_baseline": [outcome_baseline[city] for city in cities_list],
    "outcome_treatment": [outcome_treatment[city] for city in cities_list],
    "outcome_peaktime": [outcome_peaktime[city] for city in cities_list],
}))
print()
print()

# Generate sythetic data.
data_df = generate_synthetic_data(
    cities_list,
    num_days=10,
    switchback_hour=12,
    outcome_std=outcome_std,
    outcome_baseline=outcome_baseline,
    outcome_treatment=outcome_treatment,
    outcome_peaktime=outcome_peaktime,
)

print("Synthetic data")
print("--------------")
print()
display(data_df)
print()
print()

# Visualize the data that was generated.
visualize_synthetic_data(data_df)

## 3. Analysis using Mixed Linear Models

To do inference and quantify uncertainty around the effect estimates obtained
for the switchback, we use the distribution of effect estimates obtained in a
bootstrap randomization test based comparison.

Simply, how "likely" or "unlikely" is it that we observed the switchback
results due to chance?

We randomly reshuffle the day/time blocks in our data and create "fake"
switchback experiments. The estimated "fake effects" here are just due to day
by day randomness in the patterns of traffic in these cities.

If the switchback really did something and it was unlikely that we observed
the effects due to pure chance, the effects estimated in the switchback must
be relatively "extreme" compared to those in the "fake" switchback tests.

This is a version of a permutation based randomization test to do inference.

We keep track of the results we estimate in the "fake" switchback tests, and
compare the distribution of fake estimates against the actual effect estimate
from the real experiment.

### 3.1 Utility functions

#### 3.1.1 MLM point estimate functions

The following code block contains functions to estimate mixed linear models using various specifications, and aggregating the estimates to construct city-level switchback effect estimate parameters. Each method generally performs two operations:

1. **Model calibration:** An appropriate model, as defined at the start of this section, is constructed and fitted to the provided data using Powell's conjugate direction method.
2. **Effect size estimation:** Once the model has been fit, the effect size of different combinations of input features of interest are extracted from the fitted model, and an F-test is conducted to determine whether the effect is significant.


In [ ]:
def run_peaktime_mlm_hypo_tests(
    input_df: pd.DataFrame,
    outcome_var: str,
) -> pd.DataFrame:
  """Run hypothesis test for peaktime interactions.

  Args:
    input_df: input dataframe containing hourly city-level speeds, volume, and
      emission data
    outcome_var: the variable whose effect size we are attempting to measure

  Returns:
    dataframe containing output effect sizes and P-values
  """
  # ========================================================================== #
  #                             Model calibration                              #
  # ========================================================================== #

  # Define the functional form of the model.
  formula_mlm = (
      f"{outcome_var} ~ "
      "str_weekday * is_peak + "
      "switchback_on * is_peak"
  )

  # Create the model.
  md = smf.mixedlm(
      formula_mlm,
      re_formula="~ local_hour + I(local_hour**2) + I(local_hour**3) ",
      groups=input_df["city"],
      data=input_df,
  )

  # Fit the model.
  res_mlm = md.fit(method=["powell"])

  # ========================================================================== #
  #                           Effect size estimation                           #
  # ========================================================================== #

  # Get the fixed-effects parameter estimates from the model.
  param_est_df = res_mlm.fe_params

  # Baseline effect size, i.e. the effect of turning on switchback control
  # irrespective of other features.
  baseline_avg = "switchback_on[T.True]"
  baseline_effect = list(
      param_est_df[param_est_df.index.isin([baseline_avg])])[0]

  # Effect size of turning switchback on at peak hours.
  peak_param = f"switchback_on[T.True]:is_peak[T.True]"
  peak_effect = list(param_est_df[param_est_df.index.isin([peak_param])])[0]

  # Loop across all possible combinations of the tested features.
  results_by_city_peak = []
  peaktime_coefs = np.unique(input_df["is_peak"])
  for peaktime_status in peaktime_coefs:
    # Collect the point estimate for this combination of features, as well as
    # the hypothesis for which an F-test must be conducted.
    if not peaktime_status:
      # Non peak hours
      point_estimate = baseline_effect
    else:
      # Peak hours
      point_estimate = baseline_effect + peak_effect

    # Average value for the outcome variable when switchback is off.
    no_sb_avg = np.mean(
        input_df[
            (input_df.is_peak == peaktime_status)
            & (input_df.switchback_on == False)
        ][outcome_var]
    )

    # Add effect size and estimated P-values from F-test to the output.
    results_by_city_peak.append({
        "peaktime_status": peaktime_status,
        "effect_estimate": point_estimate,
        "effect_pct_baseline": point_estimate / no_sb_avg * 100,
        "p_value": 0.0,  # Unused.
    })

  # Convert to dataframe.
  results_by_city_peak = pd.DataFrame(results_by_city_peak)

  return results_by_city_peak


def run_city_mlm_hypo_tests(
    input_df: pd.DataFrame,
    outcome_var: str,
) -> pd.DataFrame:
  """Run hypothesis test for city and peaktime interactions.

  Coefficient for the non-interacted switchback_on[T.True] is the effect for the
  category that is automatically dropped.

  For this model, the coefficient on the uninteracted "switchback" parameter
  corresponds to non-highway cities, during non-peak hours. So
  dropped_category_city=Atlanta and dropped_category_peaktime=False.

  Args:
    input_df: input dataframe containing hourly city-level speeds, volume, and
      emission data
    outcome_var: the variable whose effect size we are attempting to measure

  Returns:
    dataframe containing output effect sizes and P-values
  """
  # ========================================================================== #
  #                             Model calibration                              #
  # ========================================================================== #

  # Define the functional form of the model.
  formula_mlm = (
    f"{outcome_var} ~ "
    "str_weekday * is_peak + "
    "switchback_on * is_peak * city"
  )

  # Create the model.
  md = smf.mixedlm(
      formula_mlm,
      re_formula="~ local_hour + I(local_hour**2) + I(local_hour**3) ",
      groups=input_df["city"],
      data=input_df,
  )

  # Fit the model.
  res_mlm = md.fit(method=["powell"])

  # ========================================================================== #
  #                           Effect size estimation                           #
  # ========================================================================== #

  # Get the fixed-effects parameter estimates from the model.
  param_est_df = res_mlm.fe_params

  # Baseline effect size, i.e. the effect of turning on switchback control
  # irrespective of other features.
  baseline_avg = "switchback_on[T.True]"
  baseline_effect = list(
      param_est_df[param_est_df.index.isin([baseline_avg])])[0]

  # Effect size of turning switchback on at peak hours.
  peak_param = f"{baseline_avg}:is_peak[T.True]"
  peak_effect = list(param_est_df[param_est_df.index.isin([peak_param])])[0]

  # Loop across all possible combinations of the tested features.
  results_by_city_peak = []
  city_coefs = np.unique(input_df["city"])
  peaktime_coefs = np.unique(input_df["is_peak"])
  for city_name in city_coefs:
    for peaktime_status in peaktime_coefs:

      # Collect the point estimate for this combination of features, as well as
      # the hypothesis for which an F-test must be conducted.
      if city_name == "atlanta" and not peaktime_status:
        # Global switchback parameter, applies to atlanta, non-peak
        point_estimate = baseline_effect
        hypotheses_tested = f"({baseline_avg} = 0)"
      elif city_name == "atlanta":
        # Atlanta, peak hours
        point_estimate = baseline_effect + peak_effect
        hypotheses_tested = f"({baseline_avg} + {peak_param} = 0)"
      elif city_name != "atlanta" and not peaktime_status:
        # Non Atlanta, non peak hours
        hwy_param = f"{baseline_avg}:city[T.{city_name}]"
        hwy_effect = list(param_est_df[param_est_df.index.isin([hwy_param])])[0]
        point_estimate = baseline_effect + hwy_effect
        hypotheses_tested = f"({baseline_avg} + {hwy_param} = 0)"
      else:
        # Hwy city, peak hours
        hwy_param = f"{baseline_avg}:city[T.{city_name}]"
        hwy_effect = list(param_est_df[param_est_df.index.isin([hwy_param])])[0]
        peak_hwy_param = (
            f"{baseline_avg}:is_peak[T.{peaktime_status}]:city[T.{city_name}]")
        peak_hwy_effect = (
            list(param_est_df[param_est_df.index.isin([peak_hwy_param])])[0])
        point_estimate = (
            baseline_effect + peak_effect + hwy_effect + peak_hwy_effect)
        hypotheses_tested = (
            f"({baseline_avg} + {peak_param} + {hwy_param} + {peak_hwy_param} "
            "= 0)")

      # Average value for the outcome variable when switchback is off.
      no_sb_avg = np.mean(
          input_df[
              (input_df.city == city_name)
              & (input_df.is_peak == peaktime_status)
              & (input_df.switchback_on == False)
          ][outcome_var]
      )

      # We need to conduct an F-test to see the effect is significant for the
      # city-type/peakhour combination.
      f_test_results = res_mlm.f_test(hypotheses_tested)
      f_test_val = float(f_test_results.pvalue)

      # Add effect size and estimated P-values from F-test to the output.
      results_by_city_peak.append({
          "city": city_name,
          "peaktime_status": peaktime_status,
          "effect_estimate": point_estimate,
          "effect_pct_baseline": point_estimate / no_sb_avg * 100,
          "p_value": f_test_val,
      })

  # Convert to dataframe.
  results_by_city_peak = pd.DataFrame(results_by_city_peak)

  return results_by_city_peak

#### 3.1.2 MLM permutation bootstrap inference functions

The following code block contains the utility functions aimed at performing inference. In particular, this code block contains the `conduct_randomization_based_inference` method, which acts as the primary front-facing functions used to evaluate effects sizes and P-values within the following subsection.


In [ ]:
def get_boot_sample(
    exp_df: pd.DataFrame,
    bootstrap_replace: bool = True,
    group_columns: list[str] = ["city", "local_weekday", "local_hour"],
    group_switchback_status_by_date: bool = False,
) -> pd.DataFrame:
  """Generate a bootstrap sample with randomly labeling switchback status.

  Args:
    exp_df: input dataframe containing hourly city-level speeds, volume, and
      emission data
    bootstrap_replace: whether to perform bootstrap replacement (otherwise the
      switchback status of the original dataframe is modified)
    group_columns: columns within the dataframe used to group samples together
      during the bootstrap sampling procedure. The number of rows with identical
      values in the column is preserved after resampling.
    group_switchback_status_by_date: Whether to assign all samples from the same
      switchback period with the same switchback status. If set to False, each
      (city, hour) pair is randomly assigned a switchback status.

  Returns:
    generated dataframe
  """
  boot_sample = exp_df.copy().reset_index(drop=True)

  if bootstrap_replace:
    # Create a bootstrapped dataset in which the number of samples containing
    # the same values in `group_columns` is preserved.
    boot_sample["group_id"] = boot_sample.groupby(group_columns).ngroup()
    sample_sizes = boot_sample.groupby("group_id").size()
    sb_off_df = boot_sample.copy()
    sampling_f = lambda data: np.random.choice(
        data.index,
        sample_sizes[data["group_id"].iloc[0]],
        replace=True,
    )
    boot_samp_index = (sb_off_df.groupby("group_id")).apply(sampling_f)
    boot_obs_index = [
        obs for samp_subgroup in boot_samp_index for obs in samp_subgroup]
    boot_sample = boot_sample.iloc[boot_obs_index, :]

  # Relabel the switchback status of each datapoint randomly. Note that an equal
  # number of switchback on and off samples are provided after this operation.
  boot_sample.switchback_on = False
  if group_switchback_status_by_date:
    on_dates_by_city = {}
    for city in boot_sample.city.unique():
      unique_dates = boot_sample.num_switchback.unique()
      on_dates = random.sample(unique_dates, len(unique_dates) // 2)
      on_dates_by_city[city] = on_dates
    for i in boot_sample.index:
      boot_sample.at[i, "switchback_on"] = (
          boot_sample.at[i, "num_switchback"] in
          on_dates_by_city[boot_sample.at[i, "city"]])
  else:
    for i in boot_sample.index:
      boot_sample.at[i, "switchback_on"] = random.random() > 0.5

  return boot_sample


def conduct_boot_test(
    input_df: pd.DataFrame,
    outcome_var: str,
    boot_rep: int = 100,
    method: str = "peak_city",
    boot_replace: bool = True,
    group_switchback_status_by_date: bool = False,
) -> pd.DataFrame:
  """Run the bootstrap test.

  Args:
    input_df: input dataframe containing hourly city-level speeds, volume, and
      emission data
    outcome_var: the variable whose effect size we are attempting to measure
    boot_count: the number of bootstrap samples to generate and estimate the
      effect size across
    method: the type of hypothesis test to perform. One of:
      * "peak": peak-time interactions
      * "peak_city": city / peak-time interactions
    boot_replace: whether to perform bootstrap replacement (otherwise only the
      switchback status of the original dataframe is modified)
    group_switchback_status_by_date: Whether to assign all samples from the same
      switchback period with the same switchback status. If set to False, each
      (city, hour) pair is randomly assigned a switchback status.

  Returns:
    dataframe containing output effect sizes and P-values for each bootstrap
      sample generated
  """
  placebo_test_df = pd.DataFrame()
  for _ in tqdm.tqdm(range(boot_rep)):
    # Generate a bootstrap sample.
    boot_df = get_boot_sample(
        input_df,
        boot_replace,
        group_switchback_status_by_date=group_switchback_status_by_date,
    )

    # Run hypothesis tests on the bootstrap sample.
    if method == "peak":
      # Run hypothesis test for peak-time interactions.
      results_boot_samp = run_peaktime_mlm_hypo_tests(boot_df, outcome_var)
    elif method == "peak_city":
      # Run hypothesis test for city / peak-time interactions.
      results_boot_samp = run_city_mlm_hypo_tests(boot_df, outcome_var)
    else:
      raise ValueError(f"Incorrect input: method = {method}")

    # Add results from this placebo test to the output.
    placebo_test_df = pd.concat(
        [placebo_test_df, results_boot_samp], ignore_index=True
    )

  return placebo_test_df


def conduct_randomization_based_inference(
    input_df: pd.DataFrame,
    outcome_var: str,
    use_existing_placebo: Optional[pd.DataFrame] = None,
    boot_count: int = 100,
    boot_replace: bool = True,
    method: str = "peak_city",
    group_switchback_status_by_date: bool = False,
    plot_placebo: bool = False,
) -> pd.DataFrame:
  """Run the randomization-based inference process.

  This method runs the other utility functions to compute the point estimate
  from the original dataset and that of a number bootstrap-randomized datasets.
  The two here are then combined to compute corrected P-values for the point
  effects.

  Args:
    df: input dataframe containing hourly city-level speeds, volume, and
      emission data
    outcome_var: the variable whose effect size we are attempting to measure
    use_existing_placebo: a dataframe containing results for interaction point
      estimates for different different variants of the dataset. If set to None,
      the dataframe is generated by estimating the effects from different
      bootstrap-randomized samples instead.
    boot_count: the number of bootstrap samples to generate and estimate the
      effect size across
    boot_replace: whether to perform bootstrap replacement (otherwise only the
      switchback status of the original dataframe is modified)
    method: the type of hypothesis test to perform. One of:
      * "peak": peak-time interactions
      * "peak_city": city / peak-time interactions
    group_switchback_status_by_date: Whether to assign all samples from the same
      switchback period with the same switchback status. If set to False, each
      (city, hour) pair is randomly assigned a switchback status.
    plot_placebo: whether to plot the placebo test results

  Returns:
    dataframe containing output effect sizes and P-values
  """
  # Estimate the effect size for the original dataset, without any relabeling of
  # control and treatment samples.
  if method == "peak":
    # Run hypothesis test for peak-time interactions.
    effect_estimates = run_peaktime_mlm_hypo_tests(input_df, outcome_var)
  elif method == "peak_city":
    # Run hypothesis test for city / peak-time interactions.
    effect_estimates = run_city_mlm_hypo_tests(input_df, outcome_var)
  else:
    raise ValueError(f"Incorrect input: method = {method}")

  if boot_count == 0:
    # Return the effect sizes without any P-values (used for quick iterations).
    return effect_estimates.drop(columns=["p_value"])

  # Estimate the effect size across a number of placebo experiments.
  if use_existing_placebo is None:
    placebo_results_df = conduct_boot_test(
        input_df,
        outcome_var,
        boot_count,
        method,
        boot_replace,
        group_switchback_status_by_date=group_switchback_status_by_date,
    )
  else:
    placebo_results_df = use_existing_placebo

  reported_estimates = effect_estimates.copy()
  reported_estimates = reported_estimates.drop(columns=["p_value"])
  reported_estimates["corrected_p_val"] = 0

  for i,row in reported_estimates.iterrows():
    result_row = row.copy()

    # Collect the placebo and true effect sizes.
    if method == "peak":
      peak_status = result_row.peaktime_status
      point_estimate = result_row.effect_estimate
      placebo_effects = placebo_results_df[
          (placebo_results_df.peaktime_status == peak_status)
      ].effect_estimate
    elif method == "peak_city":
      city_name = result_row.city
      peak_status = result_row.peaktime_status
      point_estimate = result_row.effect_estimate
      placebo_effects = placebo_results_df[
          (placebo_results_df.city == city_name)
          & (placebo_results_df.peaktime_status == peak_status)
      ].effect_estimate
    else:
      raise ValueError(f"Incorrect input: method = {method}")

    # Since positive effects fuel emissions are negative, we invert the effect
    # estimate before computing the corrected P-values.
    if outcome_var == "fuel_consumption_l_100k":
      point_estimate = -point_estimate
      placebo_effects = -placebo_effects

    # The corrected P-value is computed as the 1 minus the fraction of placebo
    # effect sizes found to be smaller than the true estimated effect size.
    reported_estimates.loc[i, "corrected_p_val"] = 1 - (
        stats.percentileofscore(placebo_effects, point_estimate) / 100)

  # Plot the distribution of placebo point estimates and the true point estimate
  # from the placebo study.
  if plot_placebo:
    plot_placebo_test(
        placebo_results_df, reported_estimates, method, outcome_var)

  return reported_estimates


def plot_placebo_test(
    placebo_df: pd.DataFrame,
    reported_df: pd.DataFrame,
    method: str,
    outcome_var: str,
) -> None:
  """Plot the distribution of placebo estimates and the true estimate.

  This helps us visualize how off distribution the predicted effect is from
  randomized effects.

  Args:
    placebo_df: point estimates from the placebo study for different randomized
      datasets
    reported_df: point estimates from the true dataset
    method: the type of hypothesis test to perform
    outcome_var: the variable whose effect size we are attempting to measure
  """
  units = (
      "km/hr" if outcome_var in ["speed_kph", "speed_kph_length_weighted"]
      else "L/100km")

  if method == "peak":
    _, ax = plt.subplots(1, 2, figsize=(20, 4))

    for i, peak_status in enumerate([False, True]):

      ax[i].hist(
          placebo_df[
              (placebo_df.peaktime_status == peak_status)].effect_estimate,
          bins=40,
          color=[191/255, 191/255, 242/255],
          edgecolor=[0, 0, 0.8],
      )

      # Plot the true effect size.
      predicted = list(reported_df[
          (reported_df.peaktime_status == peak_status)].effect_estimate)[0]
      predicted_pct = list(reported_df[
          (reported_df.peaktime_status == peak_status)].effect_pct_baseline)[0]
      pvalue = list(reported_df[
          (reported_df.peaktime_status == peak_status)].corrected_p_val)[0]

      ax[i].axvline(
          x=predicted,
          c="k",
          linestyle="--",
          linewidth=2,
          label=(
              f"Predicted effect = {round(predicted, 3)} {units} "
              f"({predicted_pct:.3g} %)\n"
              f"P-value = {round(pvalue, 3)}"
          ),
      )

      ax[i].set_title("Peak" if peak_status else "Off-peak", fontsize=16)
      ax[i].set_xlabel(f"Effect size ({units})", fontsize=16)
      ax[i].set_ylabel("Number of placebos", fontsize=16)
      ax[i].tick_params(axis="both", which="major", labelsize=16)
      ax[i].tick_params(axis="both", which="minor", labelsize=16)
      ax[i].legend(fontsize=14)
      ax[i].spines["top"].set_color("white")
      ax[i].spines["bottom"].set_color("white")
      ax[i].spines["right"].set_color("white")
      ax[i].spines["left"].set_color("white")
      ax[i].axhline(y=0, linewidth=2, color="k")
      ax[i].grid(False)
      ax[i].grid(axis="y", linestyle=":")

  elif method == "peak_city":
    cities = sorted(placebo_df.city.unique())

    # Generate different plots for different peak status.
    for peak_status in [False, True]:
      ncols = 5
      nrows = math.ceil(len(cities) / 5)
      _, ax = plt.subplots(nrows, ncols, figsize=(6 * ncols, 6 * nrows))

      for i, city in enumerate(cities):
        row = i // ncols
        col = i % ncols
        cityname = " ".join(x.capitalize() for x in city.split("_"))

        # Plot the distribution of placebo effect sizes.
        ax[row, col].hist(
            placebo_df[
                (placebo_df.city == city) &
                (placebo_df.peaktime_status == peak_status)].effect_estimate,
            bins=40,
            color=[191/255, 191/255, 242/255],
            edgecolor=[0, 0, 0.8],
        )

        # Plot the true effect size.
        predicted = list(reported_df[
            (reported_df.city == city) &
            (reported_df.peaktime_status == peak_status)].effect_estimate)[0]
        predicted_pct = list(reported_df[
            (reported_df.city == city) &
            (reported_df.peaktime_status == peak_status)].effect_pct_baseline)[0]
        pvalue = list(reported_df[
            (reported_df.city == city) &
            (reported_df.peaktime_status == peak_status)].corrected_p_val)[0]

        ax[row, col].axvline(
            x=predicted,
            c="k",
            linestyle="--",
            linewidth=2,
            label=(
                f"Predicted effect = {round(predicted, 3)} {units} "
                f"({predicted_pct:.3g} %)\n"
                f"P-value = {round(pvalue, 3)}"
            ),
        )

        ax[row, col].set_title(f"{cityname}, Peak={peak_status}", fontsize=16)
        ax[row, col].set_xlabel(f"Effect size ({units})", fontsize=16)
        ax[row, col].set_ylabel("Number of placebos", fontsize=16)
        ax[row, col].tick_params(axis="both", which="major", labelsize=16)
        ax[row, col].tick_params(axis="both", which="minor", labelsize=16)
        ax[row, col].legend(fontsize=14)
        ax[row, col].spines["top"].set_color("white")
        ax[row, col].spines["bottom"].set_color("white")
        ax[row, col].spines["right"].set_color("white")
        ax[row, col].spines["left"].set_color("white")
        ax[row, col].axhline(y=0, linewidth=2, color="k")
        ax[row, col].grid(False)
        ax[row, col].grid(axis="y", linestyle=":")

  else:
    raise ValueError(f"Incorrect input: method = {method}")

  plt.tight_layout()
  plt.subplots_adjust(wspace=0.2, hspace=0.2)

  plt.show()

### 3.2 Statistical inference

The following subsection runs the randomization-based inference test for a variety of different outcome variables and feature interactions. In each test, a total of 1000 bootstrap samples are generated to compute the corrected P-values. This number may be set to 0 if you like to quickly produce the effect sizes without inferring the uncertainty around these values.

#### 3.2.1 Outcome effect estimate by peaktime status

In [ ]:
results = conduct_randomization_based_inference(
    data_df,
    "outcome",
    boot_count=10,
    boot_replace=False,
    method="peak",
    group_switchback_status_by_date=False,
    use_existing_placebo=None,
    plot_placebo=True,
)

results

#### 3.2.2 Outcome effect estimate by peaktime status and city

In [ ]:
results = conduct_randomization_based_inference(
    data_df,
    "outcome",
    boot_count=10,
    boot_replace=False,
    method="peak_city",
    group_switchback_status_by_date=False,
    use_existing_placebo=None,
    plot_placebo=True,
)

results

## 4. Bayesian Hierarchical Model Estimates

### 4.1 Utility functions

#### 4.1.1 Data preparation and visualization

In [ ]:
@tf.function
def affine(
    x: tf.Tensor, coef: tf.Tensor, intercept: tf.Tensor = tf.zeros([])
) -> tf.Tensor:
  """`coef * x + intercept` with broadcasting."""
  coef = tf.ones_like(x) * coef
  intercept = tf.ones_like(x) * intercept
  return x * coef + intercept


def plot_traces(
    var_name: str, samples: np.ndarray | tf.Tensor, num_chains: int):
  """Plot traces for HMC chains.

  Args:
    var_name: The variable whose effect size we are attempting to measure.
    samples: Distribution of effect estimates.
    num_chains: Number of HMC chains.
  """
  if isinstance(samples, tf.Tensor):
    samples = samples.numpy()  # convert to numpy array
  _, axes = plt.subplots(1, 2, figsize=(14, 1.5), sharex="col", sharey="col")
  for chain in range(num_chains):
    axes[0].plot(samples[:, chain], alpha=0.7)
    axes[0].title.set_text("'{}' trace".format(var_name))
    sns.kdeplot(samples[:, chain], ax=axes[1], fill=False)
    axes[1].title.set_text("'{}' distribution".format(var_name))
    axes[0].set_xlabel("Iteration")
    axes[1].set_xlabel(var_name)
  plt.show()


def prepare_tfp_data(
    analysis_df: pd.DataFrame,
    outcome_name: str,
    hour_name: str = "local_hour",
    weekday_name: str = "local_weekday",
) -> dict[str, Any]:
  """Prepare data for Tensorflow Probability models.

  Args:
    analysis_df: A dataframe containing the analysis data.
    outcome_name: The variable whose effect size we are attempting to measure.
    hour_name: The name of the column in the dataframe containing the hour.
    weekday_name: The name of the column in the dataframe containing the day of
      the week.

  Returns:
    A dict containing the following elements:

    * "outcome": A tensor of observed outcomes.
    * "treat_status": A tensor of treatment status.
    * "hwy_city": A tensor mapping a city index to a binary value indicating
      whether the city implements a freeway segment selection heuristic.
    * "city_hour_vec": A tensor mapping city-hour pairs to a unique index.
    * "local_hour_vec": A tensor mapping each local hour to a unique index.
    * "city_vec": A tensor of mapping each city to a unique index.
    * "hour_index_city_mapping": A list used to indetify which index corresponds
      to which local hour in `city_hour_vec`.
    * "city_index_from_city_hour_vec": A list used to indetify which index
      corresponds to which city in `city_hour_vec`.
    * "city_hour_ids": A list of unique city-hour pairs.
    * "city_names": A list of the names of all cities.
    * "hour_names": A list of all local hours.
  """
  df = deepcopy(analysis_df)
  df.dropna(axis=0, how="any", inplace=True)

  city_names = sorted(df.city.unique())
  local_hours = sorted(df[hour_name].unique())
  local_weekdays = sorted(df[weekday_name].unique())
  hwy_citys = [
      "atlanta", "boston", "los_angeles", "new_york", "philadelphia", "seattle"]

  # Create numeric code for cities.
  df["city_code"] = df.city.astype(
      pd.api.types.CategoricalDtype(categories=city_names)
  ).cat.codes
  df_city = tf.convert_to_tensor(df["city_code"], dtype=tf.int32)

  # Create numeric code for local hour.
  df["local_hour_code"] = df[hour_name].astype(
      pd.api.types.CategoricalDtype(categories=local_hours)
  ).cat.codes
  df_local_hour = tf.convert_to_tensor(
      df["local_hour_code"], dtype=tf.int32
  )

  # Create numeric code for local weekday.
  df["weekday_code"] = df[weekday_name].astype(
      pd.api.types.CategoricalDtype(categories=local_weekdays)
  ).cat.codes

  # Create numeric code for each unique weekday-hour.
  df["weekday_hour_code"] = df.apply(
      lambda row: str(row.weekday_code) + "_" + str(row.local_hour_code), axis=1
  )
  weekday_hour = sorted(df.weekday_hour_code.unique())

  df["weekday_hour_code"] = df.weekday_hour_code.astype(
      pd.api.types.CategoricalDtype(categories=weekday_hour)
  ).cat.codes

  # Create numeric code for each unique city-weekday-hour.
  df["city_weekday_hour_code"] = df.apply(
      lambda row: str(row.city_code)
      + "_"
      + str(row.weekday_code)
      + "_"
      + str(row.local_hour_code),
      axis=1,
  )
  city_weekday_hour = sorted(df.city_weekday_hour_code.unique())

  # Which weekday-hour does each city-weekday-hour correspond to?
  df["city_weekday_hour_code"] = (
      df.city_weekday_hour_code.astype(
          pd.api.types.CategoricalDtype(categories=city_weekday_hour)
      ).cat.codes
  )

  # Create numeric code for each unique city-hour.
  df["city_hour_code"] = df.apply(
      lambda row: str(row.city_code) + "_" + str(row.local_hour_code), axis=1
  )
  city_hour = sorted(df.city_hour_code.unique())
  # Which hour does each city-hour correspond to?
  hour_index_for_cities = [int(c_h[2:]) for c_h in city_hour]
  # Which city does each city-hour correspond to?
  city_index_from_city_hour_vec = [int(c_h[:1]) for c_h in city_hour]
  # Which city-hours correspond to highway cities?
  df["city_hour_code"] = df.city_hour_code.astype(
      pd.api.types.CategoricalDtype(categories=city_hour)
  ).cat.codes

  df_city_hour_code = tf.convert_to_tensor(df["city_hour_code"], dtype=tf.int32)

  # Create numeric code for each city-weekday.
  df["city_weekday_code"] = df.apply(
      lambda row: str(row.city_code) + "_" + str(row.weekday_code), axis=1
  )
  city_weekdays = sorted(df.city_weekday_code.unique())
  df["city_weekday_code"] = df.city_weekday_code.astype(
      pd.api.types.CategoricalDtype(categories=city_weekdays)
  ).cat.codes

  # Create tensorflow tensors from the data frame.
  df_hwy_city = tf.convert_to_tensor(
      [1 if city in hwy_citys else 0 for city in city_names], dtype=tf.int32
  )
  df_switch_on = tf.convert_to_tensor(df["switchback_on"], dtype=tf.float32)
  df_outcome = tf.convert_to_tensor(df[outcome_name], dtype=tf.float32)

  return {
      "outcome": df_outcome,
      "treat_status": df_switch_on,
      "hwy_city": df_hwy_city,
      "city_hour_vec": df_city_hour_code,
      "local_hour_vec": df_local_hour,
      "city_vec": df_city,
      "hour_index_city_mapping": hour_index_for_cities,
      "city_index_from_city_hour_vec": city_index_from_city_hour_vec,
      "city_hour_ids": city_hour,
      "city_names": city_names,
      "hour_names": local_hours,
  }

#### 4.1.2 Hourly model

In [ ]:
def hierarchical_wrapper_hourly(
    outcome_vec: tf.Tensor,
    city_hour_vec: tf.Tensor,
    local_hour_vec: tf.Tensor,
    switchback_vec: tf.Tensor,
    hwy_city_vec: tf.Tensor,
    city_vec: tf.Tensor,
    hour_index_for_cities: list[int],
    city_index_from_city_hour_vec: list[int],
    n_burnin: int = 150000,
    n_total_iter: int = 200000,
    num_chains: int = 8,
    init_step_size: float = 0.75,
) -> object:
  """Hierarchical wrapper for outcome and switchback effects.

  Args:
    outcome_vec: A tensor of observed outcomes.
    city_hour_vec: A tensor mapping city-hour pairs to a unique index.
    local_hour_vec: A tensor mapping each local hour to a unique index.
    switchback_vec: A tensor of treatment status.
    hwy_city_vec: A tensor mapping a city index to a binary value indicating
      whether the city implements a freeway segment selection heuristic.
    city_vec: A tensor of mapping each city to a unique index.
    hour_index_for_cities: A list used to indetify which index corresponds to
      which local hour in `city_hour_vec`.
    city_index_from_city_hour_vec: A list used to indetify which index
      corresponds to which city in `city_hour_vec`.
    n_burnin: Burn in iterations. These are used to get the sampler going, and
      they are not included in the results.
    n_iter_total: Total number of iterations. Only (n_iter_total - n_burnin)
      steps are included in the results.
    num_chains: Number of different chains. Typically, 8.
    init_step_size: Step size for the sampler. Consider updating based on the
      scale of the outcome. For very large outcomes, try larger values.

  Returns:
    Hourly model outcome effects, consisting of the following elements:

    * "baseline_tau": Prior for the average treatment effect across hwy/ non hwy
      cities.
    * "corr_matrix_tau": Prior for correlation matrix between hourly
      observations.
    * "hourly_tau": Prior for average effect for each hour.
    * "sigma_tau_city": Prior for the variance of city level effects.
    * "tau_city": Prior for average effect for each city.
    * "sigma_city_hour_tau": Prior for the variance of city level effects.
    * "city_hour_tau": Prior for city-hour effects.
    * "sigma_hour": Prior for the variance of hour level effects.
    * "city_hourly_base": Prior for baseline city-hour outcomes in absence of
      treatment.
    * "sigma_obs": Prior for the variance of observed outcomes.
  """
  hours_count = len(np.unique(local_hour_vec))
  city_hour_count = len(np.unique(city_hour_vec))
  city_count = len(np.unique(city_vec))

  def hierarchical_model_hourly():
    """Hierarchical model for speeds and switchback effects. Sets the priors."""
    return tfd.JointDistributionNamed(
        dict(
            # Prior for the average treatment effect across hwy/ non hwy cities.
            baseline_tau=tfd.MultivariateNormalDiag(
                loc=[0.0, 0.0], scale_diag=tf.ones(2)
            ),
            # Prior for correlation matrix between hourly observations.
            corr_matrix_tau=tfd.CholeskyLKJ(
                dimension=hours_count, concentration=0.8
            ),
            # Prior for average effect for each hour.
            hourly_tau=lambda baseline_tau, corr_matrix_tau: tfd.MultivariateNormalLinearOperator(
                loc=tf.repeat(0.0, hours_count, axis=-1),
                scale=tf.linalg.LinearOperatorLowerTriangular(corr_matrix_tau),
            ),
            # Prior for the variance of city level effects.
            sigma_tau_city=tfd.HalfCauchy(loc=0.0, scale=2.5),
            # Prior for average effect for each city.
            tau_city=lambda baseline_tau, sigma_tau_city: tfd.MultivariateNormalDiag(
                loc=tf.gather(baseline_tau, hwy_city_vec, axis=-1),
                scale_diag=tf.ones(city_count)
                * sigma_tau_city[..., tf.newaxis],
            ),
            # Prior for the variance of city level effects.
            sigma_city_hour_tau=tfd.HalfCauchy(loc=0.0, scale=2.5),
            # Prior for city-hour effects.
            city_hour_tau=lambda hourly_tau, tau_city, sigma_city_hour_tau: tfd.MultivariateNormalDiag(
                loc=affine(
                    tf.ones(city_hour_count),
                    tf.gather(hourly_tau, hour_index_for_cities, axis=-1),
                )
                + tf.gather(tau_city, city_index_from_city_hour_vec, axis=-1),
                scale_diag=tf.ones(city_hour_count)
                * sigma_city_hour_tau[..., tf.newaxis],
            ),
            sigma_hour=tfd.HalfCauchy(loc=0.0, scale=1.0),
            # Prior for baseline city-hour outcomes in absence of treatment
            city_hourly_base=lambda sigma_hour: tfd.MultivariateNormalDiag(
                loc=tf.ones(city_hour_count) * 0.0,
                scale_diag=tf.ones(city_hour_count)
                * sigma_hour[..., tf.newaxis],
            ),
            # Prior for the variance of observed outcomes.
            sigma_obs=tfd.HalfCauchy(loc=0.0, scale=1.0),
            # Actual observed data.
            actual_obs=lambda sigma_obs, city_hourly_base, city_hour_tau: tfd.MultivariateNormalDiag(
                loc=switchback_vec
                * tf.gather(city_hour_tau, city_hour_vec, axis=-1)
                + tf.gather(city_hourly_base, city_hour_vec, axis=-1),
                scale_diag=tf.ones(len(local_hour_vec))
                * sigma_obs[..., tf.newaxis],
            ),
        )
    )

  @tf.function
  def hierarchical_hourly_log_prob(
      baseline_tau_vals,
      corr_matrix_tau_vals,
      hourly_tau_vals,
      sigma_tau_city_vals,
      tau_city_vals,
      sigma_city_hour_tau_vals,
      city_hour_tau_vals,
      sigma_hour_vals,
      city_hourly_base_vals,
      sigma_obs_vals,
  ):
    return hierarchical_model_hourly().log_prob(
        baseline_tau=baseline_tau_vals,
        corr_matrix_tau=corr_matrix_tau_vals,
        hourly_tau=hourly_tau_vals,
        sigma_tau_city=sigma_tau_city_vals,
        tau_city=tau_city_vals,
        sigma_city_hour_tau=sigma_city_hour_tau_vals,
        city_hour_tau=city_hour_tau_vals,
        sigma_hour=sigma_hour_vals,
        city_hourly_base=city_hourly_base_vals,
        sigma_obs=sigma_obs_vals,
        actual_obs=outcome_vec,
    )

  @tf.function
  def sample_hierarchical_hourly(num_chains, n_total_iter, num_burnin_steps):
    """HMC sampling from the hierarchical model with adaoptive step size."""
    # Set initial values for each parameter.
    # This shouldn't affect much, although chains could get stuck if the init
    # values are in a strange region.
    baseline_tau_init = tf.convert_to_tensor(
        [
            [0.0 + 0.005 * random.random(), 0.0 + 0.005 * random.random()]
            for _ in range(num_chains)
        ],
        dtype=tf.float32,
    )

    corr_matrix_tau_init = tf.convert_to_tensor(
        [
            tfd.CholeskyLKJ(
                dimension=hours_count,
                concentration=0.8,
            ).sample()
            for _ in range(num_chains)
        ],
        dtype=tf.float32,
    )

    hourly_tau_init = tf.convert_to_tensor(
        [
            tfd.MultivariateNormalLinearOperator(
                loc=affine(tf.ones(hours_count), 0.0),
                scale=tf.linalg.LinearOperatorLowerTriangular(
                    corr_matrix_tau_init[0]
                ),
            ).sample()
            for _ in range(num_chains)
        ],
        dtype=tf.float32,
    )

    sigma_tau_city_init = tf.convert_to_tensor(
        [1.00 + 0.002 * random.random() for _ in range(num_chains)],
        dtype=tf.float32,
    )

    tau_city_init = tf.convert_to_tensor(
        [
            tfd.MultivariateNormalDiag(
                loc=tf.convert_to_tensor(
                    np.repeat(0.0, city_count), dtype=tf.float32
                ),
                scale_diag=tf.ones(city_count) * sigma_tau_city_init[0],
            ).sample()
            for _ in range(num_chains)
        ],
        dtype=tf.float32,
    )

    sigma_city_hour_tau_init = tf.convert_to_tensor(
        [1.00 + 0.002 * random.random() for _ in range(num_chains)],
        dtype=tf.float32,
    )

    city_hour_tau_init = tf.convert_to_tensor(
        [
            tfd.MultivariateNormalDiag(
                loc=affine(tf.ones(city_hour_count), 0.0),
                scale_diag=tf.ones(city_hour_count) * 1.0,
            ).sample()
            for _ in range(num_chains)
        ],
        dtype=tf.float32,
    )

    sigma_hour_init = tf.convert_to_tensor(
        [1.00 + 0.002 * random.random() for _ in range(num_chains)],
        dtype=tf.float32,
    )

    city_hourly_base_init = tf.convert_to_tensor(
        [
            tfd.MultivariateNormalDiag(
                loc=affine(tf.ones(city_hour_count), 1.0),
                scale_diag=tf.ones(city_hour_count) * 5.0,
            ).sample()
            for _ in range(num_chains)
        ],
        dtype=tf.float32,
    )
    sigma_obs_init = tf.convert_to_tensor(
        [5.0 + 1.5 * random.random() for _ in range(num_chains)],
        dtype=tf.float32,
    )

    # Combining initial states for the HMC
    init_sample = [
        baseline_tau_init,
        corr_matrix_tau_init,
        hourly_tau_init,
        sigma_tau_city_init,
        tau_city_init,
        sigma_city_hour_tau_init,
        city_hour_tau_init,
        sigma_hour_init,
        city_hourly_base_init,
        sigma_obs_init,
    ]

    # Constrain standard deviations to the positive real axis. Other variables
    # are unconstrained.
    unconstraining_bijectors = [
        tfb.Identity(),  # baseline_tau
        tfb.CorrelationCholesky(),  # cor matrix tau
        tfb.Identity(),  # hourly_tau
        tfb.Exp(),  # sigma_tau_city
        tfb.Identity(),  # tau_city
        tfb.Exp(),  # sigma_city_hour_tau
        tfb.Identity(),  # city_hour_tau
        tfb.Exp(),  # sigma_hour
        tfb.Identity(),  # city_hourly_base
        tfb.Exp(),  # sigma_obs
    ]
    hmc = tfp.mcmc.DualAveragingStepSizeAdaptation(
        tfp.mcmc.HamiltonianMonteCarlo(
            target_log_prob_fn=hierarchical_hourly_log_prob,  # log prob of data
            step_size=init_step_size,  # Initial step size.
            num_leapfrog_steps=12,  # Number of leapfrog steps.
        ),
        num_adaptation_steps=int(0.6 * num_burnin_steps),  # Number of adaptation steps - 60% of the burnin.
        step_count_smoothing=10,
        decay_rate=0.5,
        reduce_fn=tfp.math.reduce_log_harmonic_mean_exp,  # Should not change this - do evaluation using the log harmonic mean of all chains.
        target_accept_prob=np.float64(0.65),
    )

    kernel = tfp.mcmc.TransformedTransitionKernel(
        inner_kernel=hmc, bijector=unconstraining_bijectors
    )

    samples, kernel_results = tfp.mcmc.sample_chain(
        num_results=n_total_iter,
        num_burnin_steps=num_burnin_steps,
        current_state=init_sample,
        kernel=kernel,
    )
    # Keep track of acceptance probs.
    acceptance_probs = tf.reduce_mean(
        tf.cast(
            kernel_results.inner_results.inner_results.is_accepted, tf.float32
        ),
        axis=0,
    )

    return samples, acceptance_probs

  samples, acceptance_probs = sample_hierarchical_hourly(
      num_chains, n_total_iter, n_burnin
  )
  h_model_names = collections.namedtuple(
      "h_model_names",
      [
          "baseline_tau",
          "corr_matrix_tau",
          "hourly_tau",
          "sigma_tau_city",
          "tau_city",
          "sigma_city_hour_tau",
          "city_hour_tau",
          "sigma_hour",
          "city_hourly_base",
          "sigma_obs",
      ],
  )
  h_named_samples = h_model_names._make(samples)

  print("Acceptance Probabilities for each chain:", acceptance_probs.numpy())

  return h_named_samples

#### 4.1.3 Pooled Model

In [ ]:
def hierarchical_wrapper_pooled(
    covariate_var: tf.Tensor,
    outcome_var: tf.Tensor,
    n_burnin: int,
    n_iter_total: int,
    num_chains: int,
    step_size: float = 0.005,
) -> object:
  """Simple pooled model, with a common treatment effect for every sample.

  Args:
    covariate_var: Covariate that we want to estimate the slope for. Typically,
      switchback status.
    outcome_var: Outcome we want to estimate the effect for.
    n_burnin: Burn in iterations. These are used to get the sampler going, and
      they are not included in the results.
    n_iter_total: Total number of iterations. Only (n_iter_total - n_burnin)
      steps are included in the results.
    num_chains: Number of different chains. Typically, 8.
    step_size: Step size for the sampler. Consider updating based on the scale
      of the outcome. For very large outcomes, try larger values.

  Returns:
    Pooled model outcome effects, consisting of the following elements:

    * "intercept": The intercept of the outcome.
    * "sb_effect": The slope of the outcome, i.e. the treatment effect size.
    * "sigma": Elements of a diagonal matrix depicting the scale of the
      covariance matrix.
  """
  def pooled_model(covariate_var):
    """Create the joint distribution function of our data and parameters"""
    return tfd.JointDistributionSequential([
        tfd.Normal(loc=40.0, scale=5),  # intercept
        tfd.Normal(loc=0.0, scale=5),  # slope - treatment effect
        tfd.HalfCauchy(loc=0.0, scale=5),  # sigma
        lambda sigma, slope, intercept: tfd.MultivariateNormalDiag(
            loc=covariate_var
            * (tf.ones_like(covariate_var) * slope[..., tf.newaxis])
            + tf.ones_like(covariate_var) * intercept[..., tf.newaxis],
            scale_diag=tf.ones_like(covariate_var) * sigma[..., tf.newaxis],
        ),
    ])

  @tf.function
  def pooled_log_prob_speed(intercept, slope, sigma):
    """Computes joint_log_prob for the outcome."""
    return pooled_model(covariate_var).log_prob(
        [intercept, slope, sigma, outcome_var]
    )

  @tf.function
  def sample_pooled(num_chains, n_iter_total, n_burnin):
    """Hamiltonian monte carlo sampling from the pooled model."""
    hmc = tfp.mcmc.SimpleStepSizeAdaptation(
        tfp.mcmc.HamiltonianMonteCarlo(
            target_log_prob_fn=pooled_log_prob_speed,
            step_size=step_size,
            num_leapfrog_steps=5,  # We may want to increase, but this should work.
        ),
        num_adaptation_steps=int(0.75 * n_burnin),
        target_accept_prob=np.float64(0.75),
    )
    # Set initial values for parameters.
    initial_state = [
        tf.convert_to_tensor(
            [40.0 + 5 * random.random() for _ in range(num_chains)],
            dtype=tf.float32,
        ),
        tf.convert_to_tensor(
            [0.0 + 2 * random.random() for _ in range(num_chains)],
            dtype=tf.float32,
        ),
        tf.convert_to_tensor(
            [10.0 + 2 * random.random() for _ in range(num_chains)],
            dtype=tf.float32,
        ),
    ]

    # Constrain `sigma` to the positive real axis. Other variables are
    # unconstrained.
    unconstraining_bijectors = [
        tfb.Identity(),  # intercept
        tfb.Identity(),  # slope
        tfb.Exp(),  # sigma
    ]
    kernel = tfp.mcmc.TransformedTransitionKernel(
        inner_kernel=hmc, bijector=unconstraining_bijectors
    )

    samples, kernel_results = tfp.mcmc.sample_chain(
        num_results=n_iter_total,
        num_burnin_steps=n_burnin,
        current_state=initial_state,
        num_steps_between_results=2,
        kernel=kernel,
    )

    acceptance_probs = tf.reduce_mean(
        tf.cast(
            kernel_results.inner_results.inner_results.is_accepted, tf.float32
        ),
        axis=0,
    )

    return samples, acceptance_probs

  pooled_model_samples, pooled_acceptance_probs = sample_pooled(
      num_chains, n_iter_total, n_burnin
  )

  print("Acceptance Probabilities for each chain:",
        pooled_acceptance_probs.numpy())

  PooledModel = collections.namedtuple(
      "PooledModel", ["intercept", "sb_effect", "sigma"])

  pooled_model_samples = PooledModel._make(pooled_model_samples)._asdict()

  return pooled_model_samples

### 4.2 Statistical inference

Fill in the configurable variables below to train a model for a specific outcome.

In [ ]:
# Specify the outcome variable.
outcome_var = "outcome"

# Create tensors from dataset.
tfp_data_prepped = prepare_tfp_data(data_df, outcome_var)

Run the below cell to train the pooled model for different outcomes.


In [ ]:
# Specify training parameters.
n_burnin_iters = 7500
n_total = 10000
n_chains = 8

# Create tensors from dataset.
tfp_data_pooled = prepare_tfp_data(data_df, outcome_var)

# Train the model.
pooled_model_samples = hierarchical_wrapper_pooled(
    covariate_var=tfp_data_pooled["treat_status"],
    outcome_var=tfp_data_pooled["outcome"],
    n_burnin=n_burnin_iters,
    n_iter_total=n_total,
    num_chains=n_chains,
)

# Plot results.
plot_traces("Effect estimate",
            samples=pooled_model_samples["sb_effect"][n_burnin_iters:,],
            num_chains=n_chains)

prob_positive_effect = np.mean(
    (pooled_model_samples["sb_effect"])[n_burnin_iters:,] > 0)

print("Posterior probability of positive difference between on and off periods",
      "all cities & times:", prob_positive_effect)

Run the below cell to train the hourly model for different outcomes.

In [ ]:
# Specify training parameters.
n_burnin_iters = 20000  # Large number of iterations needed for hourly model.
n_total = 30000  # Large number of iterations needed for hourly model.
n_chains = 8

# Train the model.
hourly_model_samples = hierarchical_wrapper_hourly(
    outcome_vec=tfp_data_prepped["outcome"],
    city_hour_vec=tfp_data_prepped["city_hour_vec"],
    local_hour_vec=tfp_data_prepped["local_hour_vec"],
    switchback_vec=tfp_data_prepped["treat_status"],
    hwy_city_vec=tfp_data_prepped["hwy_city"],
    city_vec=tfp_data_prepped["city_vec"],
    hour_index_for_cities=tfp_data_prepped["hour_index_city_mapping"],
    city_index_from_city_hour_vec=tfp_data_prepped["city_index_from_city_hour_vec"],
    n_burnin=n_burnin_iters,
    n_total_iter=n_total,
    num_chains=n_chains,
    init_step_size=0.05 * np.std(tfp_data_pooled["outcome"].numpy()),
)

city_hour_tau = hourly_model_samples.city_hour_tau.numpy()

Run the below cell to view the effect estimate distribution for each city-hour pair.

In [ ]:
# Loop over every city-hour pair
results_df = []
for city_hour_index in range(city_hour_tau.shape[2]):
  # Get the ID of the city which this city-hour corresponds to.
  city_index = int(((tfp_data_prepped["city_hour_ids"])[city_hour_index])[:1])
  # Get the ID of the hour which this city-hour corresponds to.
  hour_index = int(((tfp_data_prepped["city_hour_ids"])[city_hour_index])[2:])

  # Get posterior draws for this city-hour from hourly_model_samples.
  flat_hmc_draw = np.reshape(
      city_hour_tau[n_burnin_iters:, :, city_hour_index],
      [(n_total - n_burnin_iters) * n_chains],
  )

  # Excluding burnin samples. Flatten the result from multiple chains to a
  # single array.
  lbound = np.percentile(flat_hmc_draw, 2.5)  # Lower credible bound
  ubound = np.percentile(flat_hmc_draw, 97.5)  # Upper credible bound
  results_city_time = {
      # What is the name of this city?
      "city": tfp_data_prepped["city_names"][city_index],
      # What hour does this hour ID correspond to?
      "local_hr": tfp_data_prepped["hour_names"][hour_index],
      # Keep draws that are within credible bounds.
      "hmc_draws": flat_hmc_draw[
          (flat_hmc_draw > lbound) & (flat_hmc_draw < ubound)],
  }
  results_df.append(pd.DataFrame(results_city_time))

results_df = pd.concat(results_df, ignore_index=True)

# Hourly results for each city
for focus_city in (tfp_data_prepped["city_names"]):
  city_hourly_results = results_df[results_df.city == focus_city]
  print(focus_city)
  vln_plt = sns.violinplot(
      data=city_hourly_results, x="local_hr", y="hmc_draws", cut=0.0, inner=None
  )
  vln_plt.set(
      xlabel="Hour of day",
      ylabel="Effect Estimate (+km/h)", # Example for speed.
      title="Effects on " + focus_city,
  )
  plt.show()

Run the below cell to view the effect estimate distribution for each city and for peak and off-peak hours. You can specify the desired peak hours using the `peak_hours` variable, and can plot the distribution is percentage or absolute terms using the `plot_as_percentage` variable.

In [ ]:
# The hours considered as peak hours.
peak_hours = [8, 9, 15, 16, 17, 18]
# Whether to plot the figure as a percentage or in absolute terms.
plot_as_percentage = True

# Loop over every city-hour pair.
data = collections.defaultdict(lambda: collections.defaultdict(list))
for city_hour_index in range(city_hour_tau.shape[2]):
  # Get the ID of the city which this city-hour corresponds to.
  city_index = int(((tfp_data_prepped["city_hour_ids"])[city_hour_index])[:1])
  # Get the ID of the hour which this city-hour corresponds to.
  hour_index = int(((tfp_data_prepped["city_hour_ids"])[city_hour_index])[2:])

  # Get posterior draws for this city-hour from hourly_model_samples.
  flat_hmc_draw = np.reshape(
      city_hour_tau[n_burnin_iters:, :, city_hour_index],
      [(n_total - n_burnin_iters) * n_chains],
  )

  # Excluding burnin samples. Flatten the result from multiple chains to a
  # single array.
  city = tfp_data_prepped["city_names"][city_index]
  local_hour = tfp_data_prepped["hour_names"][hour_index]
  data[city]["Entire day"].append(list(flat_hmc_draw))
  if local_hour in peak_hours:
    data[city]["Peak hours"].append(list(flat_hmc_draw))

  data["all_cities"]["Entire day"].append(list(flat_hmc_draw))
  if local_hour in peak_hours:
    data["all_cities"]["Peak hours"].append(list(flat_hmc_draw))

results_df_city = []
results_df_all_cities = []
for city in ["atlanta", "boston", "chicago", "los_angeles", "miami", "new_york",
             "philadelphia", "salt_lake_city", "san_francisco", "seattle",
             "all_cities"]:
  print()
  print(city)
  print("-" * len(city))
  for peak_status in data[city].keys():
    if city == "all_cities":
      no_sb_avg = np.mean(
          data_df[
              (data_df.is_peak == (peak_status == "Peak hours")) &
              (data_df.switchback_on == False)
          ][outcome_var]
      )
    else:
      no_sb_avg = np.mean(
          data_df[
              (data_df.city == city) &
              (data_df.is_peak == (peak_status == "Peak hours")) &
              (data_df.switchback_on == False)
          ][outcome_var]
      )

    # Compute average across hours.
    hmc_draws = np.mean(np.array(data[city][peak_status]), axis=0)

    # Keep draws that are within credible bounds.
    lbound = np.percentile(hmc_draws, 2.5)  # Lower credible bound
    ubound = np.percentile(hmc_draws, 97.5)  # Upper credible bound
    median = np.median(hmc_draws)
    p_positive = len(hmc_draws[hmc_draws > 0]) / len(hmc_draws) * 100
    data[city][peak_status] = hmc_draws[
        (hmc_draws > lbound) & (hmc_draws < ubound)]

    print(f"{peak_status},",
          f"Lower bound: {lbound:.3g} ({lbound/no_sb_avg*100:.3g}%),",
          f"Median: {median:.3g} ({median/no_sb_avg*100:.3g}%),",
          f"Upper bound: {ubound:.3g} ({ubound/no_sb_avg*100:.3g}%),",
          f"P-Positive: {p_positive:.3g}%")

    for hmc_draws in data[city][peak_status]:
      (results_df_all_cities
       if city == "all_cities" else results_df_city).append({
          "city": " ".join([x.capitalize() for x in city.split("_")]),
          "peak_status": peak_status,
          "hmc_draws": hmc_draws,
          "hmc_draws_pct": hmc_draws / no_sb_avg * 100,
      })

results_df_city = pd.DataFrame(results_df_city)
results_df_all_cities = pd.DataFrame(results_df_all_cities)

print()

_, ax = plt.subplots(
    1, 2,
    # layout="constrained",
    figsize=(6, 6),
    gridspec_kw={
        "width_ratios": [3, 1],
        "wspace": 0,
        "hspace": 0,
    },
)

my_pal = {"Peak hours": [1, 0.6, 0.6], "Entire day": [0.6, 0.6, 1]}

vln_plt = sns.violinplot(
    data=results_df_city,
    x="city",
    y="hmc_draws_pct" if plot_as_percentage else "hmc_draws",
    cut=0.0,
    inner=None,
    hue="peak_status",
    palette=my_pal,
    ax=ax[0],
    legend=False,
)
vln_plt = sns.violinplot(
    data=results_df_all_cities,
    x="city",
    y="hmc_draws_pct" if plot_as_percentage else "hmc_draws",
    cut=0.0,
    inner=None,
    hue="peak_status",
    palette=my_pal,
    ax=ax[1],
)

ax[0].set_ylabel("Effect estimate distribution (%)", fontsize=13)
ax[0].tick_params(axis="y", which="major", labelsize=13)
ax[0].spines["left"].set_linewidth(1.5)
ax[0].spines["left"].set_color("black")
ax[1].spines["left"].set_color("none")
ax[1].legend([], [], frameon=False)
ax[1].set_ylabel("")
ax[1].set_yticks([])
ax[1].set_facecolor((0.9, 0.9, 0.9))
ax[0].legend(
    loc="lower left", bbox_to_anchor=(0.05, 0.05), borderaxespad=0, fontsize=13)
for i in range(2):
  ax[i].set_xlabel("")
  ax[i].set_xticklabels(
      ax[i].get_xticklabels(), rotation=45, ha="right", fontsize=13)
  ax[i].spines["top"].set_color("none")
  ax[i].spines["bottom"].set_color("black")
  ax[i].spines["bottom"].set_linewidth(1.5)
  ax[i].spines["right"].set_color("none")
  ax[i].axhline(y=0, color="black", linestyle="dashed")
  ax[i].grid(False)
  ax[i].set_ylim(*ax[0].get_ylim())

plt.show()